# A10 Master Notebook: Money-Loser Screen for US Theatrical Films

Shared, final notebook for the team submission (Matt Sorensen, Tanner Edwards). It follows **CRISP-DM**: each phase is a section, and every data decision is stated once with its evidence. Exploration and debate live in the personal notebooks (`Sorensen-A10.ipynb`, `Edwards-A10.ipynb`); only agreed work is promoted here (via PR).

**How to run (from a fresh clone)**

| Step | Command / notebook | Output |
|---|---|---|
| 1 | `python tools/get_data.py` | raw files in `data/raw/` (Wikidata snapshots are committed) |
| 2 | `notebooks/00_build_dataset.ipynb` (Run All) | `data/clean/films_unified.csv`, `reviews_unified.csv` |
| 3 | `notebooks/01_clean_dataset.ipynb` (Run All) | `data/clean/films_clean.csv`, `films_clean_columns.csv` |
| 4 | **this notebook** (Run All) | reads `films_clean.csv` only; writes nothing |

The pipeline notebooks do all merging and cleaning, so anyone can rebuild the data; this notebook shows *what* was done and *why*.

| CRISP-DM phase | Section |
|---|---|
| 1. Business Understanding | §1 |
| 2. Data Understanding (EDA) | §2 |
| 3. Data Preparation (cleaning) | §3 |
| 4. Modeling | §4 (to come) |
| 5. Evaluation | §5 (to come) |
| 6. Deployment | §6 (to come) |

## 1. Business Understanding

From the Purpose Contract (`A10_purpose_contract_FROZEN_2026-11-17.md`) with amendments A1–A7, agreed by Tanner and Matt on 2026-11-24.

| Field | Answer |
|---|---|
| **Decision** | Before a film's release, decide which projects get **deep review** (comparable films, a second reader, renegotiated terms) and which get standard review. The screen never rejects a project on its own. |
| **Action** | Each year the 20% of films with the highest predicted risk go to deep review. |
| **Inputs allowed** (A4, A6) | Anything the studio **controls before release**: budget, genre, runtime, release date, MPAA rating, director/writer/company track records (earlier films only), distributor, country, language, franchise, source material. Nothing known only after release (box office, reviews, scores, votes). |
| **Population** (A1, A2, A7) | Feature films **released in the US**, 2000–2023, budget ≥ $1M, after the cleaning rules in §3. Sensitivity rerun at a $100K floor. |
| **Label** (A3, A7) | **Money-loser = US box office < 1.25 × budget.** 1.25× US ≈ the industry rule of thumb of 2.5× worldwide (the US is about half of worldwide box office). It's a convention, not a data finding; rerun at 1× and 1.5×. |
| **Splits** (A5) | Train 2000–2015, test 2016–2019, stress test 2020–2023 (COVID era) reported separately. |
| **Baseline and bar** | Beat a genre × budget-tier rule by ≥ 5 percentage points in capture rate at 20%, with a bootstrap 95% CI excluding 0. |

**Amendments in one line each:** A1 keep 2000–2023 · A2 keep $1M floor, add $100K sensitivity · A3 2.5× is a convention (superseded by A7) · A4 inputs re-opened · A5 keep the COVID stress test · A6 timing rule = controlled before release · A7 label on US box office at 1.25×, population = released in the US.

## 2. Data Understanding

### 2a. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

CLEAN = Path("../data/clean")
films = pd.read_csv(CLEAN / "films_clean.csv", low_memory=False)
cols = pd.read_csv(CLEAN / "films_clean_columns.csv")
films["release_date"] = pd.to_datetime(films["release_date"], format="mixed", errors="coerce")

BLUE, ORANGE, GREEN, INK2 = "#2a78d6", "#eb6834", "#1baf7a", "#52514e"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "axes.titlesize": 10, "axes.titlelocation": "left",
                     "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8})

usable = films["usable"]
pop = films[films["in_population"]].copy()
pop["split"] = pd.cut(pop["year"], [1999, 2015, 2019, 2023], labels=["train 2000-15", "test 2016-19", "stress 2020-23"])
print(f"films_clean.csv: {len(films):,} films x {films.shape[1]} columns")
print(f"usable (2000-2023, budget >= $1M, revenue > 0): {usable.sum():,}   population (§3): {len(pop):,}")

### 2b. Sources and how they were joined

Six sources, assembled into one row per film by `00_build_dataset.ipynb` (key `film_uid`):

| Source | Adds | Join |
|---|---|---|
| TMDB (Kaggle, 1.5M rows) | base table, budget, revenue, genres, runtime, companies, countries, language, keywords | — |
| The Numbers (Kaggle scrape) | budget, **US and worldwide box office**, MPAA, source material, creative type, franchise | title + year matcher (hand-checked 50/50) + looser second pass |
| Box Office Mojo (GitHub scrape) | US box office, distributor | title + year |
| Rotten Tomatoes, two Kaggle datasets | director, writer, distributor, rating, critic reviews | title + year + runtime (hand-checked 0/50 wrong) |
| Wikidata snapshots (committed, CC0) | budgets; director, writer, runtime, "based on" | TMDB id |

How many outside sources (The Numbers, Box Office Mojo, the two RT datasets) confirm each usable film:

In [ ]:
src = films.loc[usable, "n_sources"].value_counts().sort_index()
display(pd.DataFrame({"usable films": src, "%": (100 * src / usable.sum()).round(1)}).rename_axis("outside sources").T)

**Finding:** 14% of usable films exist only in TMDB. TMDB is user-edited, and these are mostly junk entries (fan edits, TV episodes, a "$800M" 5-minute film) whose budget and revenue can't be checked. Handled in §3d.

### 2c. Missing values in the inputs (usable films, before filling)

The `_missing` flags record which inputs no source had, before any fill rule in §3.

In [ ]:
flags = [c for c in films.columns if c.endswith("_missing")]
miss = pd.DataFrame({"input": [c.removesuffix("_missing") for c in flags],
                     "% missing (usable)": [round(100 * films.loc[usable, c].mean(), 1) for c in flags],
                     "% missing (population)": [round(100 * pop[c].mean(), 1) for c in flags]})
display(miss.sort_values("% missing (usable)", ascending=False).set_index("input"))
tier = pd.cut(films["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
print("Missing rate by budget tier (usable films): coverage depends on budget, so blanks aren't random")
display(films.loc[usable, ["mpaa_rating_missing", "source_material_missing", "creative_type_missing", "distributor_missing"]]
        .groupby(tier[usable], observed=True).mean().mul(100).round(0))

### 2d. The inputs (population)

Distributions of each input on the final population (§3e), so the EDA describes the films the model will see.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(13, 7))
ax[0, 0].hist(np.log10(pop["fin_budget"]), bins=40, color=BLUE); ax[0, 0].set(title="Budget (log10 $; 7 = $10M)", ylabel="films")
ax[0, 1].hist(pop["runtime_min"].dropna(), bins=40, color=BLUE); ax[0, 1].set(title="Runtime (minutes)")
pop["release_date"].dt.month.value_counts().sort_index().plot.bar(ax=ax[0, 2], color=BLUE); ax[0, 2].set(title="Release month")
g = pop["genres_all"].dropna().str.split(",").explode().str.strip().value_counts()
g.sort_values().plot.barh(ax=ax[1, 0], color=BLUE); ax[1, 0].set(title="Genres (a film can have several)")
pop["mpaa_rating"].value_counts().plot.bar(ax=ax[1, 1], color=BLUE); ax[1, 1].set(title="MPAA rating (blank filled as NR)")
pop["source_group"].value_counts().plot.bar(ax=ax[1, 2], color=BLUE); ax[1, 2].set(title="Source material group")
for a in ax.flat:
    a.tick_params(axis="x", rotation=0)
ax[1, 1].tick_params(axis="x", rotation=0); ax[1, 2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

us = pop["countries"].fillna("").str.contains("United States")
print(f"Films including the US as a production country: {us.mean():.0%}; English-language: {pop['original_language'].eq('en').mean():.0%}; "
      f"in a franchise: {pop['franchise'].ne('(none)').mean():.0%}")

In [ ]:
def per_value(col, sep=","):
    v = pop[col].dropna().astype(str).str.split(sep).explode().str.strip()
    vc = v[v.ne("")].value_counts()
    b = pd.cut(vc, [0, 1, 4, 9, np.inf], labels=["1 film", "2-4", "5-9", "10+"])
    return b.value_counts().sort_index()

print("How many population films each person / company appears in (track records will be thin):")
display(pd.DataFrame({c: per_value(c) for c in ["director", "writer", "production_companies", "distributor"]}).T)

**Findings from the inputs** (detail: `Edwards-A10.ipynb` §6–7):
- **Budget** is very skewed (median about $15M, a long tail to $400M+); use it on a log scale or in tiers.
- **Runtime** is tight (middle half about 95–120 min) once non-films are removed (§3d).
- **Genres:** 19, mostly 2–3 per film; Drama and Comedy dominate.
- **People and companies:** most directors, writers and companies appear in only one film, so they work as experience counts or groups, not as names (see `FEATURE_IDEAS.md`).
- **Country and language** largely repeat each other; a US / non-US flag carries most of it.
- **Creative type and source material** are blank far more often for small films (The Numbers covers big films better).

### 2e. The label

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].hist(np.log10(pop["us_multiple"].clip(lower=1e-3)), bins=50, color=BLUE)
ax[0].axvline(np.log10(1.25), color=ORANGE, lw=1.5); ax[0].set(title="US box office / budget (log10; line = 1.25x)", ylabel="films")
by_year = pop.groupby("year")["loser"].mean()
ax[1].plot(by_year.index, 100 * by_year.values, color=BLUE, marker="o", ms=3); ax[1].set(title="Loser rate by year (%)", ylim=(0, 100))
ax[1].axvspan(2019.5, 2023.5, color=GREEN, alpha=.12)
tier_p = pd.cut(pop["fin_budget"], [0, 5e6, 20e6, 60e6, np.inf], labels=["$1-5M", "$5-20M", "$20-60M", "$60M+"])
(100 * pop.groupby(tier_p, observed=True)["loser"].mean()).plot.bar(ax=ax[2], color=BLUE); ax[2].set(title="Loser rate by budget tier (%)", ylim=(0, 100))
ax[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print(f"Losers: {pop['loser'].mean():.1%} at 1.25x; {pop['us_multiple'].lt(1).mean():.1%} at 1x; {pop['us_multiple'].lt(1.5).mean():.1%} at 1.5x")

### 2f. Data-quality problems found (and where they're handled)

| Problem | Evidence | Handled in |
|---|---|---|
| Junk entries that exist only in TMDB | 930 usable films in no outside source: fan edits, TV episodes, impossible budgets | §3d `tmdb_only` |
| Non-films: TV series, video games, TV movies, shorts, re-release editions | runtime 999 (*God of War*), TV Movie genre, "Extended Edition" titles | §3d |
| "Worldwide" revenue that is really the US gross | 618 films; 89–98% labeled losers vs 62–70% | label switched to US box office (A7) |
| TMDB budget typos (extra zero) | 103 films where TMDB and The Numbers differ > 2× (*Awake* $86M vs $8.6M) | §3b |
| Same-title mix-ups | *A Christmas Carol* (2009) had a TV version's director; late-2023 films matched junk entries | build fix (late 2023) and §3c |
| Missing values depend on budget | creative type 64% blank for $1–5M films vs 9% for $60M+ | `_missing` flags, §3c fills |

## 3. Data Preparation

All steps run in `01_clean_dataset.ipynb`; this section reports what they did. Nothing is deleted: excluded films keep an `exclude_reason`, and `in_population` marks the model's films.

### 3a. One column per fact

Each fact came from several sources. Each gets one combined column, filled from the fullest source first; the source used is in `<column>_src`. Only real values count (TMDB's 0 = unknown).

In [ ]:
src_cols = ["runtime_min", "genres_all", "mpaa_rating", "director", "writer", "production_companies", "distributor",
            "countries", "original_language", "domestic_gross"]
rows = []
for c in src_cols:
    vc = pop[c + "_src"].value_counts(normalize=True)
    rows.append({"column": c, "% filled (population)": round(100 * pop[c].notna().mean(), 1),
                 "sources used (share of population)": ", ".join(f"{k} {v:.0%}" for k, v in vc.items())})
display(pd.DataFrame(rows).set_index("column"))

### 3b. Budget repair

When TMDB's budget and The Numbers' (or Wikidata's) differ by more than 2×, The Numbers' value is used.

In [ ]:
display(films.loc[usable, "fin_budget_src"].value_counts().rename("usable films").to_frame())

### 3c. Filling missing values

Decided input by input (Tanner, Edwards §9):
- **Director, writer, runtime:** filled from Wikidata (agrees with ours 96–97% where both exist). Where our director disagreed with Wikidata, ours came from a different film with the same title, so Wikidata's replaces it (`director_mismatch`).
- **Source material:** Wikidata "based on", then grouped into `source_group` (Comic / Book / Real life / Remake / Other adaptation / Original), filled last from TMDB keywords (74% agreement in a test; origin kept in `source_group_src`).
- **MPAA:** blank → NR. **Country:** blank → main country of the language. **Franchise:** blank → `(none)`.
- **Creative type, distributor:** left blank with a `_missing` flag.

In [ ]:
fill = {c: pop[c + "_src"].isin(["wikidata", "wikidata (replaced mismatch)"]).sum() for c in ["director", "writer", "runtime_min"]}
fill["source_material"] = pop["source_material_src"].eq("wikidata").sum()
display(pd.Series(fill, name="population films filled from Wikidata").to_frame())
print(f"Directors replaced because of a same-title mix-up: {pop['director_mismatch'].sum()}")
display(pd.crosstab(pop["source_group"].fillna("(blank)"), pop["source_group_src"].fillna("(blank)"), margins=True))

### 3d. Exclusion rules

Each usable film gets the first reason that applies (rules and examples: `Edwards-A10.ipynb` §8; definitions: `01_clean_dataset.ipynb` §12).

In [ ]:
ex = films.loc[usable & films["exclude_reason"].notna()]
summary = ex.groupby("exclude_reason").agg(films=("film_uid", "size"),
                                            examples=("title", lambda s: "; ".join(s.head(4))))
display(summary.sort_values("films", ascending=False))
funnel = pd.Series({"films in films_clean.csv": len(films), "usable (2000-23, budget >= $1M, revenue > 0)": usable.sum(),
                    "after exclusion rules": (usable & films["exclude_reason"].isna()).sum(),
                    "released in the US = population": len(pop)}, name="films")
display(funnel.to_frame())

### 3e. The prepared dataset

In [ ]:
split = pop.groupby("split", observed=True).agg(films=("film_uid", "size"), losers=("loser", "mean"))
split["losers"] = (100 * split["losers"]).round(1).astype(str) + "%"
display(split)
inputs = cols[cols["group"].eq("input") & ~cols["column"].str.endswith("_missing")]
display(inputs[["column", "% filled (population)"]].set_index("column").T)
print("Model inputs = the columns above, fin_budget, and the _missing flags. Labels: loser (and us_multiple for sensitivity).")

## 4. Modeling

To come. Planned: baseline (genre × budget-tier rule) vs model, trained on 2000–2015, features from §3e plus track records built from earlier films only (`FEATURE_IDEAS.md`).

## 5. Evaluation

To come: capture rate at 20% on 2016–2019 with bootstrap CI; stress test 2020–2023; sensitivity at 1× / 1.5× and the $100K floor (A2).

## 6. Deployment

To come.